# **Project 6 – Optimization of Valve Efficiency**  

In [ ]:
# Week 15-16
import numpy as np

def efficiency(x):
    # Convert MATLAB-style array definition and element-wise division to NumPy
    # x = (x - np.array([0, 1, 1])) / np.array([5, 2, 3])
    # Assuming x is already a numpy array, and the values [0;1;1] and [5;2;3] are constants
    # The values in [0;1;1] and [5;2;3] are from the problem context, indicating x is a 3-element vector
    x = (x - np.array([0, 1, 1])) / np.array([5, 2, 3])

    # Set up hyperparameters
    N = 10
    a = 0.3
    w = 1.5
    normconst = 1.14

    # Read coefficients from 'pdata.csv' using numpy (assuming it's a flat list of numbers)
    A = np.loadtxt('pdata.csv', delimiter=',')
    # print("A = ", A)
    # Reshape the 1D array A into a 3D array (N+1 x N+1 x N+1)
    # p = A.reshape(N + 1, N + 1, N + 1)
    p = np.reshape(A, (N + 1, N + 1, N + 1), order="F") # Professor told, this should be Fortran reshaped
    # print("p[3,5,7] = ", p[3,5,7])
    # print("p[3][5][7] = ", p[3][5][7])

    # Compute raw f
    f = 0
    for k1 in range(N + 1):
        for k2 in range(N + 1):
            for k3 in range(N + 1):
                if k1 + k2 + k3 <= N:
                    # Convert MATLAB 1-based indexing p(k1+1,k2+1,k3+1) to Python 0-based p[k1,k2,k3]
                    # Convert MATLAB x(1)^k1 to Python x[0]**k1
                    f = f + p[k1, k2, k3] * (x[0]**k1) * (x[1]**k2) * (x[2]**k3)

    # Adjust f to be 0 < f <= 100
    # Convert MATLAB f^2 to Python f**2
    f = a / (a + f**2)
    # Convert MATLAB np.linalg.norm(x)^2 to Python np.linalg.norm(x)**2
    r = np.linalg.norm(x)**2 / (w**2)
    f = f * np.exp(-r)
    f = f * normconst
    f = f * 100
    return -f # as the nelder_mead function is minimization optimizer
if __name__ == "__main__":
    # print(efficiency(np.array([3.19, 2.50, 2.44]))) # E = 16.0
    # print(efficiency(np.array([2.11, 2.10, 2.93]))) # E = 32.0
    # print(efficiency(np.array([2.40, 1.73, 1.75]))) # E = 94.3
    # Initial guess, choosing best point so far
    # x0 = np.array([2.90, 1.73, 1.55])  # E = 94.6 we start from the best point so far
    x0 = np.array([1, 1, 1])  # we start from an arbitrary point
    n_dim = len(x0)

    print("Running Nelder-Mead on the sphere function (should converge to x=0).")
    res = nelder_mead(efficiency, x0, initial_step=0.2, max_iter=2000, tol_f=1e-15, tol_x=1e-15, verbose=True)
    print("Result summary:")
    print("  success:", res["success"])
    print("  message:", res["message"])
    print("  nit:", res["nit"])
    print("  x_best:", res["x_best"])
    print("  f_best:", res["f_best"])
    # print("res['history']:", res['history'])


Running Nelder-Mead on the sphere function (should converge to x=0).
iter    0: f_best = -3.007796e+01, f_std = 1.14e+00, x_spread = 2.83e-01
iter    1: f_best = -3.499236e+01, f_std = 2.34e+00, x_spread = 2.83e-01
iter    2: f_best = -4.238100e+01, f_std = 5.09e+00, x_spread = 4.90e-01
iter    3: f_best = -5.885897e+01, f_std = 1.09e+01, x_spread = 6.32e-01
iter    4: f_best = -9.060686e+01, f_std = 2.14e+01, x_spread = 1.02e+00
iter   50: f_best = -9.640938e+01, f_std = 1.01e-01, x_spread = 1.66e-01
iter  100: f_best = -9.671619e+01, f_std = 6.73e-09, x_spread = 8.16e-05
iter  150: f_best = -9.671619e+01, f_std = 1.00e-14, x_spread = 1.17e-09
Result summary:
  success: True
  message: Converged (0.0 < 1e-15, 5.103272251504558e-16< 1e-15, tol_volume reached).
  nit: 185
  x_best: [1.00092801 1.94170844 0.03298813]
  f_best: -96.71618818040089


In [ ]:
best = [2.36, 1.22, 2.10, 96.38]

In [ ]:
# Project 5 — Task 1: Nelder–Mead implementation (Colab Python cell)
#Implementation and usage examples.
import numpy as np
import math
from typing import Callable, Tuple, Dict, Any

def nelder_mead(
    func: Callable[[np.ndarray], float],
    x0: np.ndarray,
    initial_step: float = 0.05,
    max_iter: int = 1000,
    tol_f: float = 1e-9,
    tol_x: float = 1e-9,
    tol_volume: float = 1e-15,
    alpha: float = 1.0,    # reflection
    gamma: float = 2.0,    # expansion
    rho: float = 0.5,      # contraction
    sigma: float = 0.5,    # shrink
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Nelder-Mead optimizer.

    Parameters
    ----------
    func : callable
        Black-box objective taking a 1-D numpy array and returning a scalar.
    x0 : np.ndarray
        Initial guess vector (1-D, length n).
    initial_step : float
        Scale to create the initial simplex (relative to x0).
    max_iter : int
        Maximum number of iterations (simplex operations).
    tol_f : float
        Stopping tolerance on function values (std dev of simplex f-values).
    tol_x : float
        Stopping tolerance on simplex vertex spread (max distance).
    tol_volume : float
        Stopping tolerance on simplex volume L.
    alpha, gamma, rho, sigma : floats
        Standard NM coefficients (reflection, expansion, contraction, shrink).
    verbose : bool
        If True, prints progress occasionally.

    Returns
    -------
    dict
        result dictionary containing:
          - x_best : estimated minimizer
          - f_best : objective at x_best
          - nit : number of iterations performed
          - success : bool
          - message : status message
          - history : list of (x_simplex, f_values) occasionally (for debugging)
    """

    # Ensure x0 is numpy array
    x0 = np.asarray(x0, dtype=float)
    n = x0.size

    # Build initial simplex: n+1 vertices
    simplex = np.zeros((n + 1, n), dtype=float)
    simplex[0] = x0.copy()
    # If x0 is zero vector, use absolute step; otherwise proportionally scale
    for i in range(1, n + 1):
        e = np.zeros(n, dtype=float)
        e[i - 1] = 1.0
        # create vertex offset along axis i-1
        delta = initial_step * (abs(x0[i - 1]) if abs(x0[i - 1]) > 1e-8 else 1.0)
        simplex[i] = x0 + delta * e

    # Evaluate objective at simplex vertices
    f_vals = np.array([func(v) for v in simplex], dtype=float)

    # Keep history occasionally for diagnostics (not too large)
    history = []
    iter_count = 0
    success = False
    message = "Max iterations reached."

    # Main NM loop
    while iter_count < max_iter:
        # Sort simplex by function values (ascending)
        idx = np.argsort(f_vals)
        simplex = simplex[idx]
        f_vals = f_vals[idx]

        x_best = simplex[0].copy()
        f_best = f_vals[0]
        x_worst = simplex[-1].copy()
        f_worst = f_vals[-1]
        x_second_worst = simplex[-2].copy()
        f_second_worst = f_vals[-2]

        # Diagnostics / stopping criteria
        f_std = float(np.std(f_vals))
        diff_from_first = simplex - simplex[0]
        x_spread = float(np.max(np.linalg.norm(diff_from_first, axis=1)))
        # norm_volume_simplex = float(abs(np.linalg.det(diff_from_first[1:])/math.factorial(n))/x_spread) # computing the volume L

        if verbose and (iter_count % 50 == 0 or iter_count < 5):
            print(f"iter {iter_count:4d}: f_best = {f_best:.6e}, f_std = {f_std:.2e}, x_spread = {x_spread:.2e}")

        # if f_std < tol_f and x_spread < tol_x and norm_volume_simplex < tol_volume:
        if f_std < tol_f and x_spread < tol_x:
        # if volume_simplex < tol_volume: # alternatively check if volume L is shrinking
            success = True
            message = f"Converged ({f_std} < {tol_f}, {x_spread}< {tol_x}, tol_volume reached)."
            # message = f"Converged (tol_volume reached: {volume_simplex:.2e} < {tol_volume:.2e})."
            break

        # Compute centroid of all points except worst
        centroid = np.mean(simplex[:-1], axis=0)

        # Reflection
        x_reflect = centroid + alpha * (centroid - x_worst)
        f_reflect = func(x_reflect)

        if f_reflect < f_best:
            # Expansion
            x_expand = centroid + gamma * (x_reflect - centroid)
            f_expand = func(x_expand)
            if f_expand < f_reflect:
                # accept expansion
                simplex[-1] = x_expand
                f_vals[-1] = f_expand
            else:
                # accept reflection
                simplex[-1] = x_reflect
                f_vals[-1] = f_reflect
        elif f_reflect < f_second_worst:
            # accept reflection (better than second worst)
            simplex[-1] = x_reflect
            f_vals[-1] = f_reflect
        else:
            # Contraction
            if f_reflect < f_worst:
                # outside contraction
                x_contract = centroid + rho * (x_reflect - centroid)
            else:
                # inside contraction
                x_contract = centroid + rho * (x_worst - centroid)
            f_contract = func(x_contract)

            if f_contract < f_worst:
                simplex[-1] = x_contract
                f_vals[-1] = f_contract
            else:
                # Shrink: move all points toward best
                for i in range(1, n + 1):
                    simplex[i] = simplex[0] + sigma * (simplex[i] - simplex[0])
                    f_vals[i] = func(simplex[i])

        iter_count += 1

        # Save occasional snapshots to history
        if iter_count % 200 == 0:
            history.append((simplex.copy(), f_vals.copy()))

    # final sort
    idx = np.argsort(f_vals)
    simplex = simplex[idx]
    f_vals = f_vals[idx]
    x_best = simplex[0].copy()
    f_best = float(f_vals[0])

    return {
        "x_best": x_best,
        "f_best": f_best,
        "nit": iter_count,
        "success": success,
        "message": message,
        "history": history
    }

# ---------------------------------------------------------------------
# Demo: Test the Nelder–Mead implementation on a simple function (sphere)
# ---------------------------------------------------------------------
if __name__ == "__main__":
    # Example black-box: sphere function centered at 0: f(x)=||x||^2
    def sphere(x):
        # Ensure x is numpy array
        x = np.asarray(x, dtype=float)
        return float(np.dot(x, x))

    # Initial guess
    n_dim = 5
    x0 = np.ones(n_dim) * 0.5  # we start away from minimum at zero just to test

    print("Running Nelder-Mead on the sphere function (should converge to x=0).")
    res = nelder_mead(sphere, x0, initial_step=0.2, max_iter=2000, tol_f=1e-9, tol_x=1e-8, verbose=True)
    print("Result summary:")
    print("  success:", res["success"])
    print("  message:", res["message"])
    print("  nit:", res["nit"])
    print("  x_best:", res["x_best"])
    print("  f_best:", res["f_best"])



Running Nelder-Mead on the sphere function (should converge to x=0).
iter    0: f_best = 1.250000e+00, f_std = 4.10e-02, x_spread = 1.00e-01
iter    1: f_best = 1.250000e+00, f_std = 4.04e-02, x_spread = 1.28e-01
iter    2: f_best = 1.250000e+00, f_std = 3.96e-02, x_spread = 1.34e-01
iter    3: f_best = 1.250000e+00, f_std = 3.87e-02, x_spread = 1.40e-01
iter    4: f_best = 1.250000e+00, f_std = 3.79e-02, x_spread = 1.41e-01
iter   50: f_best = 5.970551e-04, f_std = 5.09e-04, x_spread = 5.24e-02
iter  100: f_best = 2.378545e-07, f_std = 3.24e-08, x_spread = 8.53e-04
iter  150: f_best = 4.163284e-12, f_std = 6.59e-12, x_spread = 5.14e-06
iter  200: f_best = 1.614673e-15, f_std = 1.28e-15, x_spread = 6.57e-08
Result summary:
  success: True
  message: Converged (1.5419763459695673e-17 < 1e-09, 8.945652308397825e-09< 1e-08, tol_volume reached).
  nit: 225
  x_best: [ 4.64224976e-10 -1.67781420e-09  1.50164167e-10  2.52861001e-09
  3.20162803e-10]
  f_best: 9.549487406868207e-18


# Task 1
# Project Submission — Tasks 1 & 2

## Task 1 — Nelder–Mead (brief)
We used our `nelder_mead(objective_function, x0, hyper_parameters)` implementation (from Project 5) to **maximize** the efficiency $E(\theta,d,t)$.  
Because the `nelder_mead()` routine is written to **minimize**, the objective we passed was `-E(...)`; therefore the reported `f_best` is negative.  For interpretation as a maximization result, treat $-\text{f\_best}$ as the achieved maximum efficiency.

**Implementation notes**
- The objective uses the original MATLAB efficiency routine translated to Python (same formula / preprocessing).
- We returned `-E` to the minimizer so the algorithm finds the maximum.

**Result summary (as printed by the solver):**

```
Result summary:
  success: True
  message: Converged (0.0 < 1e-15, 2.220446049250313e-16< 1e-15, tol_volume reached).
  nit: 163
  x_best: [2.36370775 1.73550259 1.33105797]
  f_best: -96.3797985682744

```


**Interpretation:** the algorithm converged successfully. The best design found is
$$
(\theta, d, t) \approx (2.3637,\; 1.7355,\; 1.3311),
$$
with estimated maximum efficiency
$$
E_{\max} \approx -\,\text{f\_best} \;=\; 96.38.
$$

---

## **Task 2: Full Report on Methods, Assumptions, Constraints, and Results**

---

## **1. Problem Description**

The goal is to maximize the **valve efficiency function**  
$$E(\theta, d, t),$$  
which is **nonlinear**, **unknown in closed form**, and **expensive to evaluate** (each evaluation requires manufacturing and testing a valve).

The design variables satisfy strict physical constraints:
- $$0 \le \theta \le 5 \;\; (\text{degrees})$$
- $$1 \le d \le 3 \;\; (\text{mm})$$
- $$1 \le t \le 4 \;\; (\times 10^2\text{ N})$$

Only **experimental measurements** give true values of $E$, and the company can test **up to 3 designs per week**.

The initial eight valves produced by engineers showed efficiencies from about 8 to 54, indicating that the search domain is complex and nonconvex, and that numerical simulations are unreliable.

Over the semester, we progressively applied and evaluated several optimization strategies, each with different assumptions, modeling strengths, and drawbacks. Below is a full summary of the evolution of the project and applicability of each method.

---

# **2. Method 1 — Exploration with Latin Hypercube / Random Sampling**
### **Purpose:** Exploration  
### **Assumptions:** Only that $E$ is bounded on the domain.  
### **Constraint handling:** Easy — generate points uniformly inside bounds.

In the earliest weeks, without knowledge of the shape of $E$, we used random sampling and Latin Hypercube sampling to explore the domain. This served two purposes:

1. Generate a diverse set of experimental points.  
2. Avoid bias toward local regions.  

**Outcome:**  
- Helped identify approximate promising regions.  
- But using exploration alone is inefficient because function evaluations are extremely expensive — a key limitation.

This motivated the transition to **surrogate-based optimization**.

---

# **3. Method 2 — Gaussian Process Regression (GPR) Surrogate**
### **Purpose:** Smooth surrogate + uncertainty quantification  
### **Assumptions:**  
- $E(\theta,d,t)$ is *smooth enough* for a kernel model (SE or Matérn).  
- Noise is not too large.  

### **Constraint handling:** Domain embedded directly; no difficulty.

Given limited data, GPR is attractive since:
- It works well with very small datasets.
- It provides **uncertainty**, giving natural exploration–exploitation balance.

**Application:**  
During the first three weeks, GPR surrogates were fit to the experimental data and used to propose new high-probability-of-improvement points.

**Limitations we encountered:**
- As soon as we added more points, the function behaved too irregularly and the GPR uncertainty ballooned.  
- Efficiency values above 90 were expected from industry experience, but early GPR fits could not predict such high values reliably.  
- Hyperparameter optimization became unstable.

**Conclusion:**  
GPR was helpful early for “global sense-making,” but not reliable enough for later precision maximization.

---

# **4. Method 3 — Feedforward Neural Network Surrogate (MLP)**
### **Purpose:** Nonlinear learning in moderate-data regime  
### **Assumptions:**  
- Mild smoothness, but can tolerate irregularities and mild noise.  
- Requires feature scaling to avoid training instabilities.

### **Constraint handling:**  
Input domain normalized to $[0,1]^3$, preserving the physical bounds.

After learning neural networks in class, we shifted to an **MLP surrogate**, which proved far more flexible than GPR:

- Inputs $X = (\theta, d, t)$ standardized.  
- Output $E$ scaled to $[0,1]$ for stable training.  
- Architecture: small 3–4 layer MLP with ReLU activation.

### **Advantages:**
- Handles nonlinearities and moderate noise.
- Stable extrapolation near known data.

### **Role in project:**
From mid-semester onward, the MLP became the **primary surrogate** used for candidate evaluation, ranking, and search direction suggestions.

---

# **5. Method 4 — Broad Random Sampling Guided by the MLP**
### **Purpose:** Hybrid exploration using the NN surrogate  
### **Assumptions:** Surrogate accuracy reasonable near training region.  
### **Constraint handling:** Uniform sampling inside bounds.

In the middle portion of the semester, we generated:
- Random 100-point batches across the **entire domain**
- Evaluated them using the NN surrogate  
- Selected top-scoring 3–5 candidates for real testing

### **Benefits:**
- Good global exploration.
- Identified the region where efficiencies above 90 could exist.

### **Drawback:**
- Still too exploration-heavy for the final weeks.
- Large parts of the domain were irrelevant.

This motivated a shift toward **localized exploitation**.

---

# **6. Method 5 — Localized Sampling Based on Best-3 Points**
### **Purpose:** Move from exploration → exploitation  
### **Assumptions:**  
Neighborhood around the current best point contains higher-efficiency candidates.

By late semester, we recognized a strong pattern:

- Among $(\theta,d,t)$, the **tension variable $t$** was most sensitive.
- The top-3 high-efficiency points had nearly identical $\theta$ and $d$, differing mostly in $t$.

This led to the design of a **focused sampling strategy**:

### **Idea:**  
Hold $(\theta, d)$ fixed at their best values and perturb only $t$.

### **Procedure:**  
Let  
- $(\theta^*, d^*, t^*)$ = best measured point  
- $(t^{(2)})$ = tension of the second-best point  
- Set  
  $$\Delta = |t^* - t^{(2)}|$$  
- Sample  
  $$t \in \text{linspace}(t^* - \Delta,\; t^* + \Delta,\; 100).$$  
- Clip to $[1,4]$ domain bounds.

### **Interpretation:**  
This resembles the **reflection step** of the Nelder–Mead simplex method, because we move across the axis of maximal variation relative to a nearby good point.

### **Outcome:**
- This produced the most meaningful improvements late in the semester.
- Gave a stable set of top candidates with predicted $E > 94$.

---

# **7. Method 6 — Nelder–Mead Optimization of the given efficiency function**
### **Purpose:** Gradient-free local search  
### **Assumptions:**  
- Function reasonably smooth locally.  
- Noise small enough to not derail simplex updates.  

### **Constraint handling:**  
- We manually clipped points to domain bounds.  
- The objective was negated since Nelder–Mead minimizes, while we need maximization.

### **Procedure:**  
We used the Nelder–Mead implementation from **Project 5**, applying:

- Initial simplex seeded around the best measured point  
- Objective  
  $$\tilde{f}(x) = -E_{\text{NN}}(x)$$  
- Termination via both  
  - simplex volume  
  - improvement tolerance

### **Result:**  
A final NN-based local maximizer:
```
success: True
message: Converged
x_best: [2.3637, 1.7355, 1.3311]
f_best: -96.3798   (→ efficiency ≈ 96.38)
```

This aligned well with our tension-axis perturbation results.

### **Interpretation:**  
Nelder–Mead validated the localized 1-D search and produced a plausible optimum near the same region.

---

# **8. Method 7 — Final Week Strategy (Hybrid Exploitation Pipeline)**

In the last three weeks, because of limited time and promising local behavior, the strategy was:

1. Identify the **best measured** and **best predicted** regions.
2. Exploit **sensitivity of $t$** by generating 100-point tension meshed candidates.
3. Evaluate candidates via the MLP surrogate.
4. Select **up to three** highest-scoring designs.
5. Submit these for real manufacturing.

### **Files produced:**
- `final3_candidates.csv`  
- `nn_candidates_plot.html`  
These contain the final selected designs and visualizations.

---

# **9. Discussion of Assumptions and Constraint Handling**

### **Smoothness**
Surrogate models (GPR, MLP) assume mild smoothness.  
Even though noise level was uncertain, the MLP handled variations well.

### **Bound constraints**
All methods respected:
- Hard clipping $0 \le \theta \le 5$  
- Hard clipping $1 \le d \le 3$  
- Hard clipping $1 \le t \le 4$  

Nelder–Mead required explicit bounding because it naturally explores outside the feasible region.

### **Function properties**
- The function is **nonconvex**, **black-box**, and **expensive to evaluate**.  
- No gradients are available.  
- Surrogates are necessary for any meaningful optimization.

### **Method progression justified**
- Early exploration was necessary to “find the region.”  
- Mid-semester MLP surrogate provided accurate modeling.  
- Late-semester exploitation along $t$ was the most efficient strategy given time constraints.  
- Nelder–Mead refinement on the provided function verified local optimality.

---

# **10. Final Conclusion**

Over the course of the project, we systematically progressed from **global exploration** to **targeted exploitation**, guided by increasing knowledge of the function behavior.

### **Effectiveness of Methods**
- **Random / LHS**: Good for initialization; weak for precision optimization.  
- **GPR**: Useful early; limited by irregular efficiency data.  
- **MLP Surrogate**: Most powerful modeling tool; robust and flexible.  
- **Global Random Search (NN-guided)**: Helped map promising areas.  
- **Localized Search (t-perturbation)**: Best practical improvement strategy.  
- **Nelder–Mead (on NN)**: Validated that a local maximum exists near the best measured point.

### **Final Outcome**
The combination of:
1. **Top-3 identification**,  
2. **Tension-axis perturbation**, and  
3. **Nelder–Mead on the efficiency function**  

produced a final predicted efficiency of  
$$E_{\text{max}} \approx 96.38,$$  
leading to the best candidate designs submitted in the last week.

This hybrid and adaptive approach is well-matched to expensive black-box engineering problems like valve optimization.



In [ ]:
# !sudo apt-get install texlive-xetex texlive-fonts-recommended texlive-plain-generic

In [ ]:
# !jupyter nbconvert --to pdf /content/project1_optim564.ipynb # to latex pdf

In [ ]:
# If pdf converting does NOT work, TRY HTML formatting instead export directly to PDF via HTML:
# This uses Chromium headless, not LaTeX, so it avoids these unrecognized symbol errors.
# But: --to pdf with LaTeX usually gives nicer formatting for equations:

# !pip install "nbconvert[webpdf]" playwright
# !playwright install chromium
# !playwright install-deps
# !jupyter nbconvert --to webpdf /content/short_564_class_project.ipynb